## Encoded VaR draft with a 2018 / 2019 split

Pre-train on Nifty levels through 31 December 2018, then fine-tune on the rest of `data/Nifty50_data.xlsx`. The primary full-sample notebook (`Encoded_VaR_FCP.ipynb`) splits at the end of 2017 instead, which is the split in the report.


In [5]:
# Earlier Encoded VaR split: pre-train, then fine-tune.
# Original Colab files (not in this repository):
#   /content/Initial weights training data.xlsx   # comment in the file: 2005 to 2018
#   /content/Fine tuning data & test.xlsx         # comment in the file: 2019 to 2024
# Both roles are filled from data/Nifty50_data.xlsx, split on 31 December 2018,
# which is the date the original comments describe. The workbook ends 25 September 2023.

# Shared dataset locator.
# The workbook and the two-year CSV live in the project data/ folder.
# Running this notebook from a method folder (Historical, GARCH, ...) or from
# the project root both resolve. Machine-specific paths from the original
# Colab / local drives are no longer required.
from pathlib import Path

def find_nifty_file(filename="Nifty50_data.xlsx"):
    """Return the first existing copy of a file under data/ or the working directory."""
    here = Path.cwd().resolve()
    candidates = [
        here / "data" / filename,
        here.parent / "data" / filename,
        here / filename,
    ]
    for path in candidates:
        if path.exists():
            return str(path.resolve())
    raise FileNotFoundError(
        f"Could not find {filename}. Open the notebook from the project folder "
        "or from its method folder so that data/ is visible."
    )

NIFTY_PATH = find_nifty_file("Nifty50_data.xlsx")
print("Using dataset:", NIFTY_PATH)

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam

def set_gpu():
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                if not tf.config.experimental.get_memory_growth(gpu):
                    tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(e)

set_gpu()

class VariationalAutoencoder:
    def __init__(self, input_dim=252, latent_dim=32, beta=1.0):
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.beta = beta
        self.build_vae()

    def build_vae(self):
        inputs = Input(shape=(self.input_dim,))
        x = Dense(64, activation='relu')(inputs)
        x = Dense(32, activation='relu')(x)

        z_mean = Dense(self.latent_dim)(x)
        z_log_var = Dense(self.latent_dim)(x)

        def sampling(args):
            z_mean, z_log_var = args
            epsilon = K.random_normal(shape=(K.shape(z_mean)[0], self.latent_dim))
            return z_mean + K.exp(0.5 * z_log_var) * epsilon

        z = Lambda(sampling)([z_mean, z_log_var])

        latent_inputs = Input(shape=(self.latent_dim,))
        x = Dense(32, activation='relu')(latent_inputs)
        x = Dense(64, activation='relu')(x)
        decoder_outputs = Dense(self.input_dim)(x)

        decoder = Model(latent_inputs, decoder_outputs, name="decoder")

        class VAE(tf.keras.Model):
            def __init__(self, encoder, decoder, beta, **kwargs):
                super().__init__(**kwargs)
                self.encoder = encoder
                self.decoder = decoder
                self.beta = beta

            def train_step(self, data):
                if len(data.shape) == 1:
                    data = tf.expand_dims(data, axis=0)

                with tf.GradientTape() as tape:
                    z_mean, z_log_var, z = self.encoder(data, training=True)
                    reconstruction = self.decoder(z, training=True)
                    reconstruction_loss = tf.reduce_mean(tf.keras.losses.mse(data, reconstruction))
                    kl_loss = -0.5 * tf.reduce_mean(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
                    total_loss = reconstruction_loss + self.beta * kl_loss

                grads = tape.gradient(total_loss, self.trainable_weights)
                self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

                return {"loss": total_loss, "reconstruction_loss": reconstruction_loss, "kl_loss": kl_loss}

        self.encoder = Model(inputs, [z_mean, z_log_var, z], name="encoder")
        self.decoder = decoder
        self.vae = VAE(self.encoder, self.decoder, self.beta)
        self.vae.compile(optimizer=Adam(learning_rate=0.001))

class EncodedVaR:
    def __init__(self, window_size=90, latent_dim=32, confidence_level=0.95, beta=1.0):
        self.window_size = window_size
        self.latent_dim = latent_dim
        self.confidence_level = confidence_level
        self.beta = beta
        self.vae = None

    def train_initial_model(self, data, epochs=50, batch_size=32):
        processed_data = self.preprocess_data(data)
        self.vae = VariationalAutoencoder(input_dim=self.window_size, latent_dim=self.latent_dim, beta=self.beta).vae

        if processed_data.shape[0] < self.window_size:
            raise ValueError(f"Not enough data to reshape into windows of size {self.window_size}. Available data: {processed_data.shape[0]}")

        processed_data = self.create_sliding_windows(processed_data, self.window_size)

        with tf.device('/GPU:0'):
            self.vae.fit(processed_data, epochs=epochs, batch_size=batch_size, verbose=1)

    def fine_tune_and_calculate_var(self, data, epochs=10, batch_size=16):
        processed_data = self.preprocess_data(data)
        processed_data = self.create_sliding_windows(processed_data, self.window_size)
        predicted_vars = []
        violations = 0

        for i in range(0, len(processed_data) - self.window_size, 21):
            # The train_data is now a sliding window of size 90
            train_data = processed_data[i:i+self.window_size]
            with tf.device('/GPU:0'):
                self.vae.fit(train_data, epochs=epochs, batch_size=batch_size, verbose=0)

            # The test data is now a sliding window of size 90
            test_data = processed_data[i+1:i+self.window_size+1]
            with tf.device('/GPU:0'):
                z_mean, z_log_var, z = self.vae.encoder.predict(test_data)
                generated_scenarios = self.generate_scenarios(z_mean, z_log_var, 1000)

            var = np.percentile(generated_scenarios, (1 - self.confidence_level) * 100)
            predicted_vars.append(var)

            if i + self.window_size < len(processed_data):
                actual_return = processed_data[i + self.window_size, -1]
                if actual_return < var:
                    violations += 1

        violation_rate = (violations / len(predicted_vars)) * 100
        print(f"Total Violations: {violations}, Violation Rate: {violation_rate:.2f}%")
        return predicted_vars

    def preprocess_data(self, data):
        returns = data.pct_change().dropna()
        rolling_mean = returns.rolling(window=self.window_size).mean()
        rolling_std = returns.rolling(window=self.window_size).std()
        normalized_data = ((returns - rolling_mean) / rolling_std).dropna().values
        return normalized_data

    def generate_scenarios(self, z_mean, z_log_var, n_scenarios):
        random_latent_points = tf.random.normal(shape=(n_scenarios, self.latent_dim))
        generated_samples = self.vae.decoder.predict(random_latent_points)
        return generated_samples[:, -1]

    def create_sliding_windows(self, data, window_size):
        return np.array([data[i:i+window_size] for i in range(len(data) - window_size + 1)])

def main():
    # Was two Colab workbooks. Equivalent split of the shared Nifty 50 file:
    # initial weights on 2005-2018, fine-tune and test from 2019 onward.
    nifty = pd.read_excel(NIFTY_PATH)
    nifty["Date"] = pd.to_datetime(nifty["Dates"])
    data = nifty.loc[nifty["Date"] <= "2018-12-31", "NIFTY Index"]

    encoded_var = EncodedVaR(window_size=90, latent_dim=32, confidence_level=0.95, beta=1.0)
    encoded_var.train_initial_model(data, epochs=50, batch_size=32)

    new_data = nifty.loc[nifty["Date"] > "2018-12-31", "NIFTY Index"]

    predicted_vars = encoded_var.fine_tune_and_calculate_var(new_data, epochs=10, batch_size=16)
    return predicted_vars

if __name__ == "__main__":
    predicted_vars = main()


Epoch 1/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - kl_loss: 0.0493 - loss: 1.1253 - reconstruction_loss: 1.0760
Epoch 2/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - kl_loss: 0.0033 - loss: 1.0480 - reconstruction_loss: 1.0447
Epoch 3/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - kl_loss: 0.0011 - loss: 1.0460 - reconstruction_loss: 1.0449
Epoch 4/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - kl_loss: 4.7219e-04 - loss: 1.0423 - reconstruction_loss: 1.0418
Epoch 5/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - kl_loss: 2.3442e-04 - loss: 1.0403 - reconstruction_loss: 1.0400
Epoch 6/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - kl_loss: 1.2876e-04 - loss: 1.0404 - reconstruction_loss: 1.0403
Epoch 7/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - kl_loss: 7.8848e-05 - loss: 1.0399 - reconstruction_loss: 1.0398
Epoch 8/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - kl_loss: 4.7366e-05 - loss: 1.0407 - reconstruction_loss: 1.0406
Epoch 9/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - kl_lo